In [ ]:
import asyncio, os, textwrap
from dotenv import load_dotenv

from agents import Agent, Runner
from agents.mcp import MCPServerStdio

load_dotenv()  

OUTPUT_DIR   = os.path.expanduser("mcp_research_output")   
REPORT_FILE  = os.path.join(OUTPUT_DIR, "ai_research_report.md")
RESEARCH_TOPIC = "top AI companies 2025 breakthroughs"
SCRAPE_URL   = "https://news.ycombinator.com"                
FETCH_URL    = "https://hacker-news.firebaseio.com/v0/topstories.json?print=pretty"

os.makedirs(OUTPUT_DIR, exist_ok=True)
print(f"Output directory: {OUTPUT_DIR}")

In [ ]:
def make_mcp_servers():
    """Return the three MCP server objects (not yet started)."""

    fs_server = MCPServerStdio(
        name="filesystem",
        params={
            "command": "npx",
            "args": ["-y", "@modelcontextprotocol/server-filesystem", OUTPUT_DIR],
        },
    )

    playwright_server = MCPServerStdio(
        name="playwright",
        params={
            "command": "npx",
            "args": ["@playwright/mcp@latest", "--headless"],
        },
    )

    fetch_server = MCPServerStdio(
        name="fetch",
        params={
            "command": "uvx",   
            "args": ["mcp-server-fetch"],
        },
    )

    return [fs_server, playwright_server, fetch_server]

print("MCP server configs ready")

In [ ]:
SYSTEM_PROMPT = textwrap.dedent("""
    You are an autonomous research analyst. You have three tool groups:

    -filesystem tools  — read_file, write_file, list_directory  (use for saving notes & reports)
    -playwright tools  — browser_navigate, browser_snapshot, browser_click,
                          browser_take_screenshot  (use for live web scraping)
    -fetch tools       — fetch  (use for lightweight HTTP requests / JSON APIs)

    WORKFLOW you must follow:
    1. Use **playwright** to navigate to the target URL and extract the top 10 story titles
       visible on the page. Use browser_snapshot to read the DOM text.
    2. Use **fetch** to call the Hacker News Firebase API and retrieve the IDs of the
       top 5 stories as JSON.
    3. Synthesise both data sources into a concise markdown report with:
       - A short executive summary (3-4 sentences)
       - A numbered list of the scraped headlines
       - A numbered list of the top 5 story IDs from the API
       - A brief analysis paragraph connecting the findings to the research topic
    4. Use **filesystem** write_file to save the report to: ai_research_report.md
    5. Confirm the file was written by reading it back with read_file and printing
       the first 200 characters.

    Be concise in tool calls. Do NOT hallucinate data — only use what the tools return.
""").strip()

print("System prompt ready")

In [ ]:
async def run_research_agent():
    mcp_servers = make_mcp_servers()

    # Boot all three MCP servers and keep them alive for the whole run
    async with mcp_servers[0], mcp_servers[1], mcp_servers[2]:

        agent = Agent(
            name="ResearchAnalyst",
            model="gpt-4o",
            instructions=SYSTEM_PROMPT,
            mcp_servers=mcp_servers,
        )

        user_message = (
            f"Research topic: '{RESEARCH_TOPIC}'.\n"
            f"Scrape URL: {SCRAPE_URL}\n"
            f"API URL: {FETCH_URL}\n"
            "Follow your workflow and produce the final report."
        )

        print("Agent running ...\n")
        result = await Runner.run(agent, user_message)

    return result


result = await run_research_agent()
print("\n" + "═" * 60)
print("FINAL AGENT OUTPUT")
print("═" * 60)
print(result.final_output)

In [ ]:
from IPython.display import Markdown, display

if os.path.exists(REPORT_FILE):
    with open(REPORT_FILE) as f:
        report_md = f.read()
    print(f"Report saved at: {REPORT_FILE}  ({len(report_md)} chars)\n")
    display(Markdown(report_md))
else:
    print("Report file not found — check agent output above for errors.")

In [ ]:
print("Tool calls made by the agent:\n")
for i, item in enumerate(result.new_items, 1):
    item_type = type(item).__name__
    if hasattr(item, 'raw_item'):
        raw = item.raw_item
        if hasattr(raw, 'type'):
            if raw.type == 'tool_call_item':
                print(f"  [{i:02d}] TOOL CALL  → {raw.name}")
            elif raw.type == 'tool_call_output_item':
                out_preview = str(raw.output)[:120].replace('\n', ' ')
                print(f"  [{i:02d}] TOOL OUTPUT → {out_preview}…")
            else:
                print(f"  [{i:02d}] {item_type}: {raw.type}")
        else:
            print(f"  [{i:02d}] {item_type}")
    else:
        print(f"  [{i:02d}] {item_type}")